In [9]:

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import MarkdownHeaderTextSplitter
"""
    第一步,源数据的提取,清洗
    然后分片切成chunk
"""
# 加载文档
with open('resources/评估.md','r',encoding='utf-8') as file:
    doc_content = file.read()

# 创建markdown分片对象
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        # 按照标题切分
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ],
    # 设置是否需要携带标题
    strip_headers=False
)

# 通过对象对文档做切片,切成chunks
doc_chunks = markdown_splitter.split_text(doc_content)
# 列表每一个是一个document对象,page_content是chunk内容

In [10]:
from pydantic import SecretStr
from dotenv import load_dotenv
import  os
from langchain_openai import OpenAIEmbeddings
"""
    第二步,创建向量模型和向量数据库对象,给向量数据库挂载向量模型,传入chunk做向量化转换,并存入数据库
    数据库会调用向量模型拿到向量值,再进行自动存入的操作
"""

# 加载环境变量
load_dotenv()

ali_base_url = os.getenv('ALI_BASE_URL')
ali_api_key = os.getenv('ALI_API_KEY','')

# 创建向量模型,这里使用阿里云的云模型
ali_embeddings = OpenAIEmbeddings(
    # 传入模型base_url
    base_url=ali_base_url,
    # api_key需要转成pydantic对象
    api_key=SecretStr(ali_api_key),
    # 指定向量模型
    model='qwen3.7-text-embedding',
    # 取消向量长度检查
    check_embedding_ctx_length=False
)

# 创建基于内存的向量数据库
vector_store = InMemoryVectorStore(ali_embeddings)

In [11]:
"""
    第三步,将chunks,切分好的文本写入向量数据库
"""

new_chunks = [
    doc_chunks[index:index+20]  # 截取下标起始往后20个长度
    for index in range(0,len(doc_chunks),20) # 下标步长为20
]
# 调用向量模型
for chunks in new_chunks:
    # 将列表套列表的形式拆出来存储
    ids = vector_store.add_documents(chunks)
    # 打印片段的存储id
    print(ids)

['3758d1cf-5ae9-4837-879f-a132290076b0', '66a64d98-4345-4775-a369-75dcf579200e', 'ca2d296d-ee32-431e-be92-f4717b5d70c9', '594a3c51-448a-4460-b93c-92c18c272b53', '7019c8e0-c167-42f5-909e-23f7a64889a9', 'ad921f3b-4db6-48b0-8431-6f8859038e7e', '02d1ce5f-244a-4050-a954-fa204925c15e', '1a219ffe-1baf-4ec7-bdd4-50cf410523e9', 'b0bb5f55-06e8-4b3b-8739-3c5d0ce42ba5', '695402ce-f2fd-4989-af07-b1986078227d', '07aaf7b5-090b-47b0-95a3-4abbae485765', '4e582c40-0a0a-42d0-80b0-f6cc8eaafc97', '7044bf50-e753-42fe-9d88-110d10a4365e', 'f0217b43-9aea-46f5-864b-21592a13d06e', '4060fe2e-4eb2-4364-ba04-9e5870ed668a', '9ba685c9-1d6f-440c-aa95-29a3640dcad0', 'e9689f84-8c23-47b7-b67d-8dc659542c8a', 'b7871771-eb3c-48d5-93a8-7fce7cb6c9db', '4cbd51ce-045d-4d9a-8f96-343e883b678b', '4ff2042b-8e08-4cee-8c7e-130d687870b0']
['fdb0479a-e3fd-45dc-8af9-325859af1800']


In [13]:
"""
    第四步,初始化agent,定义知识库检索工具进行挂载
"""
from langchain_core.tools import tool
# 先初始化模型对象
model = init_chat_model(
    model='deepseek-v4-flash',
    # 自动从环境变量识别url和读取api_key
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

# 定义知识库检索的工具函数
@tool
def retrieve_knowledge(user_prompt:str) -> str:
    """
    根据问题检索知识库,返回知识库文档
    :param user_prompt: 用户问题
    :return: 知识库相关性文档
    """
    # 将用户消息传入知识库做语义匹配
    docs = vector_store.similarity_search(user_prompt)

    # 将有效字段保留,拼成完整的大字符串发给大模型
    content = [item.page_content for item in docs]

    result = '\n\n'.join(content)

    return result



system_prompt = """
    你是一个智能助手，名叫千早爱音。
    平时可以闲聊,如果用户问你一些专业领域的问题的话:结合提供的**参考材料工具**给出清晰、准确的回答。

规则：
1. 如果参考材料中有相关信息，请基于该信息回答。
2. 如果参考材料中没有相关信息，请明确告知“当前资料未覆盖该问题”，不要自行编造。
3. 回答要简洁、有条理，必要时可以分点列出。
4. 始终保持礼貌和专业的语气。
"""
# 初始化智能体对象
agent = create_agent(
    model=model,
    tools=[retrieve_knowledge],
    # 定义系统提示词,限定不能虚构内容,在专业领域需要参考知识库
    system_prompt=system_prompt
)

In [17]:
"""
    第五步,调用大模型,传入用户输入提示词打印结果
"""
from langchain_core.messages import HumanMessage

# invoke需要传入一个字典作为参数
response = agent.invoke({
    'messages': [HumanMessage(content='晚上好呀伙计')]
})
# 打印大模型的解析结果
print(response['messages'][-1].content)

晚上好呀！我是千早爱音，很高兴见到你～有什么我可以帮忙的吗？😊
